In [1]:
#!/usr/bin/env python3
"""
Read a CSV containing printed pymatgen.Structure text with site magnetizations,
label atoms in the original structure order as Fe₁, Fe₂, Ni₁, Ni₂, ...,
and extract local neighbor shells for every atom.

Output:
- one row per central atom
- shell columns containing neighbor labels, site indices, periodic images,
  distances, species, and local magnetic moments

Install:
    pip install pandas numpy pymatgen
"""

import json
import re
import tempfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from pymatgen.core import Lattice, Structure


# ----------------- CONFIG -----------------
INPUT_CSV = "hea_dataset_with_magmom.csv"
OUTPUT_CSV = "hea_local_configurations_with_labels.csv"

MAX_SHELLS = 4
MAX_NEIGHBOR_DIST = 6.0       # Å
DISTANCE_TOLERANCE = 0.10      # Å
# ------------------------------------------


# Float pattern that also handles scientific notation like 3e-05
FLOAT_RE = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
SITE_LINE_RE = re.compile(
    rf"^\s*(\d+)\s+(\S+)\s+({FLOAT_RE})\s+({FLOAT_RE})\s+({FLOAT_RE})(?:\s+({FLOAT_RE}))?"
)


def to_subscript(n: int) -> str:
    """
    Convert an integer to unicode subscript form.
    Example: 12 -> ₁₂
    """
    sub_map = str.maketrans("0123456789-+", "₀₁₂₃₄₅₆₇₈₉₋₊")
    return str(n).translate(sub_map)


def detect_structure_column(df: pd.DataFrame) -> str:
    """
    Find the column that contains structure text.
    """
    preferred = ["structure", "struct", "structure_str", "structure_text", "cif"]
    for c in preferred:
        if c in df.columns:
            return c

    # fallback: longest text column
    lengths = []
    for c in df.columns:
        try:
            med = df[c].astype(str).str.len().median()
        except Exception:
            med = -1
        lengths.append((c, med))
    lengths.sort(key=lambda x: x[1], reverse=True)
    return lengths[0][0]


def write_temp_file(text: str, suffix: str = ".cif") -> Path:
    tf = tempfile.NamedTemporaryFile(delete=False, suffix=suffix, mode="w", encoding="utf-8")
    tf.write(text)
    tf.flush()
    tf.close()
    return Path(tf.name)


def parse_printed_structure(s: str) -> Structure:
    """
    Parse the printed pymatgen.Structure block, including the magmom column
    if present.

    This preserves the exact site order from the CSV text.
    """
    lines = [ln.rstrip() for ln in s.splitlines() if ln.strip()]

    abc = None
    angles = None

    for ln in lines:
        low = ln.strip().lower()
        if low.startswith("abc"):
            nums = re.findall(FLOAT_RE, ln)
            if len(nums) >= 3:
                abc = [float(nums[0]), float(nums[1]), float(nums[2])]
        elif low.startswith("angles"):
            nums = re.findall(FLOAT_RE, ln)
            if len(nums) >= 3:
                angles = [float(nums[0]), float(nums[1]), float(nums[2])]

    if abc is None or angles is None:
        raise ValueError("Could not parse lattice parameters (abc / angles).")

    # Find start of site table
    start_idx = None
    for i, ln in enumerate(lines):
        if "Sites (" in ln or ln.strip().startswith("#"):
            start_idx = i + 1
            break

    if start_idx is None:
        raise ValueError("Could not locate the site table in the structure text.")

    species = []
    coords = []
    magmoms = []

    for ln in lines[start_idx:]:
        if ln.strip().startswith("---") or set(ln.strip()) <= {"-"}:
            continue

        m = SITE_LINE_RE.match(ln)
        if not m:
            # Stop only if we are clearly past the site table
            continue

        _, sp, a, b, c, mag = m.groups()
        species.append(sp)
        coords.append([float(a), float(b), float(c)])
        magmoms.append(float(mag) if mag is not None else np.nan)

    if not species:
        raise ValueError("No atomic sites were parsed from the structure text.")

    lattice = Lattice.from_parameters(*abc, *angles)
    structure = Structure(
        lattice,
        species,
        coords,
        site_properties={"magmom": magmoms},
    )
    return structure


def parse_structure_cell(cell_value, base_dir: Path) -> Structure:
    """
    Parse one structure cell robustly:
      1) JSON-style pymatgen Structure dict
      2) path to CIF file
      3) raw CIF text
      4) printed Structure text
    """
    if pd.isna(cell_value):
        raise ValueError("Empty structure cell.")

    s = str(cell_value).strip()

    # 1) JSON-ish dict
    if s.startswith("{") and ("@module" in s or '"lattice"' in s):
        try:
            d = json.loads(s)
            return Structure.from_dict(d)
        except Exception:
            try:
                d = json.loads(s.replace("'", '"'))
                return Structure.from_dict(d)
            except Exception:
                pass

    # 2) Existing file path
    if s.lower().endswith(".cif") or "/" in s or "\\" in s:
        p = Path(s)
        if not p.exists():
            p2 = (base_dir / s).resolve()
            if p2.exists():
                p = p2
        if p.exists():
            return Structure.from_file(str(p))

    # 3) Raw CIF text
    if "data_" in s or "_atom_site" in s or "loop_" in s:
        tf = write_temp_file(s, suffix=".cif")
        try:
            return Structure.from_file(str(tf))
        finally:
            try:
                tf.unlink()
            except Exception:
                pass

    # 4) Printed structure text
    if "Full Formula" in s or "Sites (" in s or ("abc" in s and "angles" in s):
        return parse_printed_structure(s)

    raise ValueError("Unrecognized structure format.")


def build_site_labels(structure: Structure):
    """
    Create labels like Fe₁, Fe₂, Ni₁, Ni₂ in the exact order the sites appear.
    Returns a list of labels indexed by site index.
    """
    counts = defaultdict(int)
    labels = []

    for site in structure:
        sp = site.species_string
        counts[sp] += 1
        labels.append(f"{sp}{to_subscript(counts[sp])}")

    return labels


def group_neighbors_into_shells(neighbor_data, tol=DISTANCE_TOLERANCE):
    """
    Group neighbors into shells by distance.
    Neighbors are assumed sorted by increasing distance.
    """
    if not neighbor_data:
        return []

    neighbor_data = sorted(neighbor_data, key=lambda x: x["distance"])

    shells = []
    current_shell = [neighbor_data[0]]
    shell_ref = neighbor_data[0]["distance"]

    for nd in neighbor_data[1:]:
        if abs(nd["distance"] - shell_ref) <= tol:
            current_shell.append(nd)
        else:
            shells.append(current_shell)
            current_shell = [nd]
            shell_ref = nd["distance"]

    if current_shell:
        shells.append(current_shell)

    return shells


def extract_environment(structure: Structure, central_index: int, max_dist=MAX_NEIGHBOR_DIST, max_shells=MAX_SHELLS):
    """
    Extract neighbor shells around a central atom.
    """
    central_site = structure[central_index]
    neighbors = structure.get_neighbors(central_site, max_dist)

    neighbor_data = []
    for nb in neighbors:
        nb_index = int(nb.index)
        site = structure[nb_index]

        # Periodic image vector if available
        image = getattr(nb, "image", None)
        if image is None:
            image = getattr(nb, "jimage", None)

        if image is not None:
            image = tuple(int(x) for x in image)

        neighbor_data.append(
            {
                "site_index": nb_index,
                "distance": float(getattr(nb, "nn_distance", central_site.distance(nb))),
                "species": site.species_string,
                "magmom": float(site.properties.get("magmom", np.nan)),
                "image": image,
            }
        )

    if not neighbor_data:
        return []

    neighbor_data.sort(key=lambda x: x["distance"])
    shells_raw = group_neighbors_into_shells(neighbor_data, tol=DISTANCE_TOLERANCE)

    shell_info = []
    for shell in shells_raw[:max_shells]:
        shell_info.append(
            {
                "count": len(shell),
                "n_species": len(set(x["species"] for x in shell)),
                "site_indices": [x["site_index"] for x in shell],
                "species": [x["species"] for x in shell],
                "magmoms": [x["magmom"] for x in shell],
                "distances": [x["distance"] for x in shell],
                "images": [x["image"] for x in shell],
            }
        )

    return shell_info


def shell_to_string(values, fmt=None):
    """
    Convert shell values to a compact semicolon-separated string.
    """
    out = []
    for v in values:
        if v is None:
            out.append("NA")
        elif isinstance(v, float) and np.isnan(v):
            out.append("NA")
        elif fmt is not None and isinstance(v, (float, int, np.floating, np.integer)):
            out.append(fmt.format(v))
        else:
            out.append(str(v))
    return ";".join(out)


def process_dataframe(df: pd.DataFrame, structure_col: str, base_dir: Path) -> pd.DataFrame:
    """
    Create one row per central atom.
    """
    rows = []

    total = len(df)
    for input_row_idx, row in df.iterrows():
        try:
            structure = parse_structure_cell(row[structure_col], base_dir)
        except Exception as e:
            print(f"[WARN] Skipping row {input_row_idx}: could not parse structure -> {e}")
            continue

        site_labels = build_site_labels(structure)

        for site_idx in range(len(structure)):
            try:
                shell_info = extract_environment(structure, site_idx)

                central_site = structure[site_idx]
                central_magmom = float(central_site.properties.get("magmom", np.nan))

                out = {
                    "_input_row": int(input_row_idx),
                    "candidate_formula": row.get("composition", row.get("formula", row.get("candidate_formula", ""))),
                    "total_magnetization": row.get("total_magnetization", ""),
                    "volume": row.get("volume", ""),
                    "mag_per_volume": row.get("mag_per_volume", ""),

                    "central_site_index": int(site_idx),
                    "central_label": site_labels[site_idx],
                    "central_species": central_site.species_string,
                    "central_magmom": central_magmom,
                    "central_frac_coords": ",".join(f"{x:.6f}" for x in central_site.frac_coords),

                    "n_detected_shells": len(shell_info),
                }

                # Shell-wise columns
                for shell_no in range(1, MAX_SHELLS + 1):
                    shell_key_prefix = f"shell{shell_no}"

                    if shell_no <= len(shell_info):
                        sh = shell_info[shell_no - 1]

                        labels_with_images = []
                        labels_no_images = []

                        for idx, img in zip(sh["site_indices"], sh["images"]):
                            lab = site_labels[idx]
                            labels_no_images.append(lab)
                            if img is None:
                                labels_with_images.append(lab)
                            else:
                                labels_with_images.append(f"{lab}@{img}")

                        out[f"{shell_key_prefix}_count"] = sh["count"]
                        out[f"{shell_key_prefix}_nspecies"] = sh["n_species"]
                        out[f"{shell_key_prefix}_site_indices"] = shell_to_string(sh["site_indices"])
                        out[f"{shell_key_prefix}_labels"] = shell_to_string(labels_with_images)
                        out[f"{shell_key_prefix}_site_labels"] = shell_to_string(labels_no_images)
                        out[f"{shell_key_prefix}_species"] = shell_to_string(sh["species"])
                        out[f"{shell_key_prefix}_magmoms"] = shell_to_string(sh["magmoms"], fmt="{:.6f}")
                        out[f"{shell_key_prefix}_distances_A"] = shell_to_string(sh["distances"], fmt="{:.4f}")
                        out[f"{shell_key_prefix}_images"] = shell_to_string(sh["images"])
                    else:
                        out[f"{shell_key_prefix}_count"] = np.nan
                        out[f"{shell_key_prefix}_nspecies"] = np.nan
                        out[f"{shell_key_prefix}_site_indices"] = ""
                        out[f"{shell_key_prefix}_labels"] = ""
                        out[f"{shell_key_prefix}_site_labels"] = ""
                        out[f"{shell_key_prefix}_species"] = ""
                        out[f"{shell_key_prefix}_magmoms"] = ""
                        out[f"{shell_key_prefix}_distances_A"] = ""
                        out[f"{shell_key_prefix}_images"] = ""

                rows.append(out)

            except Exception as e:
                print(f"[WARN] Failed row {input_row_idx}, site {site_idx}: {e}")
                continue

    return pd.DataFrame(rows)


def main(input_csv_path: str = INPUT_CSV, output_csv_path: str = OUTPUT_CSV):
    base_dir = Path.cwd()
    csv_path = Path(input_csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"Input CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError("Input CSV is empty.")

    structure_col = detect_structure_column(df)
    print(f"Detected structure column: {structure_col}")

    out_df = process_dataframe(df, structure_col, base_dir)

    if out_df.empty:
        print("No rows were processed successfully. Nothing to write.")
        return

    # Put the key columns first
    first_cols = [
        "_input_row",
        "candidate_formula",
        "central_site_index",
        "central_label",
        "central_species",
        "central_magmom",
        "central_frac_coords",
        "n_detected_shells",
    ]
    remaining = [c for c in out_df.columns if c not in first_cols]
    out_df = out_df[first_cols + remaining]

    out_df.to_csv(output_csv_path, index=False)
    print(f"Saved: {output_csv_path}")
    print(f"Rows written: {len(out_df)}")


if __name__ == "__main__":
    main()

Detected structure column: structure
Saved: hea_local_configurations_with_labels.csv
Rows written: 800
